In [1]:
import os
import time
import functools
from openai import OpenAI
from dotenv import load_dotenv
import traceback

def retry(max_retries: int = 5, sleep_seconds: float = 15.0):
    """
    重试装饰器：函数报错时等待 sleep_seconds 秒后重试，最多重试 max_retries 次。
    超过重试上限后，抛出最后一次的异常。
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, max_retries + 2):  # 1次正常 + max_retries次重试
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exc = e
                    if attempt <= max_retries:
                        print(f"[retry] {func.__name__} 第{attempt}次失败: {traceback.format_exc()}，{sleep_seconds}秒后重试...")
                        time.sleep(sleep_seconds)
                    else:
                        print(f"[retry] {func.__name__} 已达最大重试次数({max_retries})，放弃。")
            raise last_exc
        return wrapper
    return decorator


@retry(max_retries=3, sleep_seconds=5.0)
def qwen(prompt, web_search: bool = False, enable_thinking: bool = False):
    client = OpenAI(
        # 从环境变量读取，或直接填写 api_key="sk-xxx"
        api_key="sk-fb07e345e2b04562ad5acf2d4bfee8fa", #os.getenv("QWEN_API_KEY"),
        # 北京地域（默认）
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"

    )

    completion = client.chat.completions.create(
        model=os.getenv("INFER_LLM_NAME", "qwen3-max"),
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        # stream=True,  # 如需流式输出，取消注释
        extra_body={
            "enable_search": web_search,  # 开启联网搜索
            "enable_thinking": enable_thinking,  # 开关思考功能
            # "thinking_budget": 81920   # 可选：限制思考过程最大Token数（默认32768）
            # "search_options": {
            #     "forced_search": True,        # 强制开启搜索（默认false，由模型判断是否使用）
            #     "search_strategy": "turbo",   # 搜索策略：turbo(默认)/max/agent/agent_max
            #     "enable_source": True,        # 返回搜索结果来源（仅部分模型支持）
            #     "enable_citation": True,      # 开启角标标注[1]样式（需enable_source为true）
            #     "citation_format": "[ref_<number>]"  # 角标样式：[<number>] 或 [ref_<number>]
            # }
        }

    )
    if enable_thinking:
        res ={
            "content": completion.choices[0].message.content,
            "reasoning_content":completion.choices[0].message.reasoning_content
        }
    else:
        res ={
            "content": completion.choices[0].message.content,
            "reasoning_content":""
        }

    return res

# 读数据

In [2]:
import pandas as pd

In [3]:
data = pd.read_excel("./316合并后债权融资专题问答库_财务_税务.xlsx", engine = "openpyxl")#.fillna("", inplace = True)
data.columns
data.rename(columns = {"序号":"id","问题":"question","答案":"answer"}, inplace = True)
data['policy'] = None
data['reasoning'] = None

In [4]:
data['policy'].fillna("暂无", inplace = True)
data['reasoning'].fillna("暂无", inplace = True)

/tmp/ipykernel_1904608/1966434329.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['policy'].fillna("暂无", inplace = True)
/tmp/ipykernel_1904608/1966434329.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try

In [5]:
data['answer'] = data['answer'] + "\n\n" + "- 政策依据"  +"\n"+ data['policy']

In [6]:
data.head()

,id,question,answer,policy,reasoning
0,1,【金融机构】 金融企业取得的小额贷款利息收入在企业所得税年度纳税申报时，应填写在《A1010...,<p>金融企业取得小额贷款利息收入，填写A101020表第7行“发放贷款及垫资”栏次。</p...,暂无,暂无
1,2,"【金融机构】银行业金融机构利润表中的""利息收入""项目，与企业所得税申报时的利息收入金额存在差...",<p>这两个表都是按照会计口径填写的，所以，表A101020“银行业利息收入”应等于《利润表...,暂无,暂无
2,3,【金融机构】我国银行设立的境外分行向境内企业提供贷款服务并从境内取得利息收入，境内企业在支付...,<p>1.与总机构汇总缴纳企业所得税。境内机构向境外分行支付利息时，不代扣代缴企业所得税；<...,暂无,暂无
3,4,【金融机构】金融企业按照会计准则计提的委托贷款损失准备金，在进行企业所得税汇算清缴时是否准予...,<p>金融企业的委托贷款不承担风险和损失，不得提取贷款损失准备金在企业所得税税前扣除。</p...,暂无,暂无
4,5,【金融机构】金融企业准予在企业所得税前提取贷款损失准备金的贷款资产范围具体包括哪些类别？,<p>准予税前提取贷款损失准备金的贷款资产范围包括：</p><p>（一）贷款（含抵押、质押、...,暂无,暂无


In [7]:
# 拆除头20条进行know-how迭代
cut = int(len(data) * 1)
print(cut)
data_train = data.loc[:cut,:]
data_test = data.loc[cut:,:]

330


# 推理迭代

In [8]:
import json5
from tqdm import tqdm
from prompts import single_v0, single_v1, compression_v0

In [9]:
eb = "合并后债券融资"
q = data['question'][0]
r = data['reasoning'][0]
a = data['answer'][0]

print(single_v1(eb,q,r,a))

# Role
你是一台独立运行的「行业知识提炼引擎」，专精于从单一业务样本中**向上抽象**出具备普适性、可复用性的行业标准 KNOW-HOW。你的输出将直接进入并发的知识库构建流水线，与其他抽取结果进行最终归并。

# Objective
基于输入的业务样本（用户行为、问题、推理链、标准答案），**独立执行一次完整的知识泛化与提炼**，生成结构化的 Markdown 格式 KNOW-HOW 片段。
- **核心任务**：剔除样本中的个性化细节，提炼出适用于该类业务场景的通用规范、底层准则或标准SOP。
- **去重策略**：无需关心其他并发节点正在抽取什么内容，专注于将当前样本提炼到最精炼、最泛化的形态，便于后续的自动去重与合并。

# Compression (质量优化指令)
在生成本次 KNOW-HOW 前，必须执行以下质量优化：

1. **升维抽象**：绝对禁止"就事论事"或简单复述标准答案。必须向上抽象一层，提炼出**通用业务准则**或**全局合规要求**。
   - ❌ 差示例："用户询问数电票如何归档，答案是保存XML格式"
   - ✅ 优示例："电子会计凭证无纸化归档的通用准则：归档文件必须包含完整数字签名的原始文件（如XML），满足条件可豁免纸质存档"

2. **结构化重组**：使用 Markdown 标题层级清晰定义知识边界。
   - 必须包含：适用范畴（Scope）、核心规则（Rules）、豁免/例外（Exceptions）、政策/依据（References，仅当标准答案明确提及时才写）

3. **原子化剥离**：确保本次抽取的 KNOW-HOW 是一个**独立的、自洽的知识单元**，不依赖于其他片段即可理解。

# Strict Constraints (抽取红线)
1. **绝对泛化原则**：`Know_How` 内容必须剔除所有当前样本的个性化数据（如具体金额、特定企业名称、个案时间等），仅保留可复用的业务逻辑。
2. **原文依从性**：所有规则必须源自 `<Standard_Answer>;` 和 `<Reasoning_Chain>;` 中明确提及的标准或事实，**严禁脑补**标准答案中未提及的政策依据。
3. **格式纯净度**：输出内容必须是可直接入库的知识片段，不含任何解释性前言（如"根据用户问题..."）或结语。


In [10]:
response = qwen(single_v1(eb,q,r,a))

In [11]:
print(json5.loads(response['content'])['Logic_Diagnosis'])

该样本为特定纳税申报表行次填写指引，仅针对金融企业小额贷款利息收入在《A101020金融企业收入明细表》中的填报位置，未体现可泛化的判断逻辑、处理流程或合规标准，属于具体操作事实陈述，无通用业务规则可抽取。


In [12]:
print(json5.loads(response['content'])['Know_How'])

In [13]:
import time
import json
import json5
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import os

file_lock = Lock()

def process_single_item_with_retry(c, data_train, eb, qwen_func, v0_func, output_file, max_retries=None):
    """
    处理单个数据项，带重试机制直到成功
    
    Args:
        max_retries: 最大重试次数，None表示无限重试直到成功（建议设置上限如100防止死循环）
    """
    q = data_train['question'][c]
    r = data_train['reasoning'][c]
    a = data_train['answer'][c]
    
    retry_count = 0
    base_wait = 3  # 基础等待秒数
    
    while True:
        # 检查是否达到最大重试次数（如果设置了的话）
        if max_retries is not None and retry_count >= max_retries:
            error_info = {
                "index": c,
                "status": "failed",
                "error": "达到最大重试次数",
                "retry_count": retry_count,
                "last_error": last_error_msg
            }
            # 带锁写入失败状态
            with file_lock:
                _update_json_file(output_file, str(c), error_info)
            print(f"[Failed] 第 {c} 项在重试 {max_retries} 次后放弃")
            return c, "failed", None
        
        try:
            if retry_count > 0:
                print(f"[Retry {retry_count}] 第 {c} 项第 {retry_count + 1} 次尝试...")
            
            # 调用模型
            response = qwen_func(v0_func(eb, q, r, a))
            
            # 解析JSON（这里可能会因为模型格式错误而失败）
            try:
                content = json5.loads(response['content'])
            except (json.JSONDecodeError, KeyError, TypeError) as json_err:
                raise Exception(f"JSON解析失败: {str(json_err)} | 原始内容: {str(response.get('content', 'N/A'))[:100]}")
            
            # 验证必需字段
            if 'Know_How' not in content:
                raise Exception(f"响应缺少必需字段 'Know_How'")
            
            know_how = content['Know_How']
            logic_diagnosis = content.get('Logic_Diagnosis', '')
            
            # 构建成功结果
            result = {
                "index": c,
                "Know_How": know_how,
                "Logic_Diagnosis": logic_diagnosis,
                "status": "success",
                "retry_count": retry_count  # 记录实际重试次数
            }
            
            # 带锁写入文件
            with file_lock:
                _update_json_file(output_file, str(c), result)
            
            status_msg = f"第 {c}/{len(data_train)} 项完成"
            if retry_count > 0:
                status_msg += f"（历经 {retry_count} 次重试）"
            print(f"[Success] {status_msg}")
            
            return c, "success", know_how
            
        except Exception as e:
            retry_count += 1
            last_error_msg = str(e)
            
            # 每5次重试打印一次详细错误，避免日志刷屏
            if retry_count % 5 == 1:
                print(f"[Error] 第 {c} 项第 {retry_count} 次失败: {last_error_msg[:150]}...")
                print(traceback.format_exc())
            else:
                print(f"[Error] 第 {c} 项第 {retry_count} 次失败: {last_error_msg[:100]}...")
            
            # 指数退避：等待时间随重试次数指数增长，但最多60秒
            wait_time = min(base_wait * (2 ** (retry_count - 1)), 60)
            print(f"[Wait] 等待 {wait_time} 秒后重试...")
            time.sleep(wait_time)

def _update_json_file(file_path, key, value):
    """辅助函数：线程安全地更新JSON文件"""
    data_dict = {}
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data_dict = json.load(f)
        except (json.JSONDecodeError, IOError):
            data_dict = {}
    
    data_dict[key] = value
    
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data_dict, f, ensure_ascii=False, indent=2)

def kh_extraction_multithread(data_train, max_workers=5, output_file="./kh_results_zqrz317.json", max_retries_per_item=100):
    """
    多线程 Know-How 提取（带无限重试机制）
    
    Args:
        max_workers: 并发线程数（根据API限流调整）
        output_file: 输出JSON文件（以行号为key）
        max_retries_per_item: 每个项目的最大重试次数，None表示真正无限重试（谨慎使用），默认100次
    """
    eb = "合并后债券融资"
    total = len(data_train)
    
    print(f"开始多线程处理（带重试），总数据量: {total}, 并发数: {max_workers}")
    if max_retries_per_item is None:
        print("⚠️  警告：设置为无限重试模式，遇到顽固错误时线程将永不退出")
    else:
        print(f"最大重试次数: {max_retries_per_item} 次/项")
    
    # 初始化输出文件（支持断点续传）
    existing_data = {}
    if os.path.exists(output_file):
        try:
            with open(output_file, 'r', encoding='utf-8') as f:
                existing_data = json.load(f)
            print(f"📂 发现已有进度文件，包含 {len(existing_data)} 条记录，自动续传")
        except:
            existing_data = {}
    
    # 筛选未完成的任务（支持断点续传）
    pending_indices = [
        c for c in range(total) 
        if str(c) not in existing_data or existing_data.get(str(c), {}).get('status') != 'success'
    ]
    
    completed = total - len(pending_indices)
    print(f"已完成: {completed}, 待处理: {len(pending_indices)}")
    
    # 主循环
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_index = {
            executor.submit(
                process_single_item_with_retry, 
                c, data_train, eb, qwen, single_v1, output_file, max_retries_per_item
            ): c for c in pending_indices
        }
        
        processed = 0
        for future in as_completed(future_to_index):
            c = future_to_index[future]
            processed += 1
            
            try:
                index, status, _ = future.result()
                if status == "success":
                    completed += 1
                    print(f"✅ 进度: {completed}/{total} ({completed/total*100:.1f}%)")
                else:
                    print(f"❌ 第 {index} 项最终失败")
            except Exception as e:
                print(f"💥 第 {c} 项处理异常: {e}")
            
            # 每完成10%打印一次汇总
            if processed % max(1, len(pending_indices) // 10) == 0:
                print(f"📊 已处理 {processed}/{len(pending_indices)} 个任务")
    
    print(f"\n🏁 全部完成！结果保存于: {output_file}")

# 使用示例：
# 严格无限重试（直到成功，慎用）
# kh_extraction_multithread(data_train, max_workers=5, max_retries_per_item=None)

# 推荐：最多重试100次（防止极端情况死循环）
kh_extraction_multithread(data_train, max_workers=5, max_retries_per_item=100)

开始多线程处理（带重试），总数据量: 330, 并发数: 5
最大重试次数: 100 次/项
已完成: 0, 待处理: 330
[Success] 第 3/330 项完成
✅ 进度: 1/330 (0.3%)
[Success] 第 0/330 项完成
✅ 进度: 2/330 (0.6%)
[Success] 第 1/330 项完成
✅ 进度: 3/330 (0.9%)
[Success] 第 4/330 项完成
✅ 进度: 4/330 (1.2%)
[Success] 第 5/330 项完成
✅ 进度: 5/330 (1.5%)
[Success] 第 2/330 项完成
✅ 进度: 6/330 (1.8%)
[Success] 第 7/330 项完成
✅ 进度: 7/330 (2.1%)
[Error] 第 9 项第 1 次失败: <string>:3 Unexpected "
" at column 40...
Traceback (most recent call last):
  File "/tmp/ipykernel_1904608/1407241259.py", line 50, in process_single_item_with_retry
    content = json5.loads(response['content'])
  File "/usr/local/lib/python3.10/dist-packages/json5/lib.py", line 108, in loads
    raise ValueError(err)
ValueError: <string>:3 Unexpected "
" at column 40

[Wait] 等待 3 秒后重试...
[Success] 第 6/330 项完成
✅ 进度: 8/330 (2.4%)
[Success] 第 8/330 项完成
✅ 进度: 9/330 (2.7%)
[Success] 第 10/330 项完成
✅ 进度: 10/330 (3.0%)
[Retry 1] 第 9 项第 2 次尝试...
[Success] 第 11/330 项完成
✅ 进度: 11/330 (3.3%)
[Success] 第 12/330 项完成
✅ 进度: 12/330 (3.

# 二级提炼：对问题中提取的一级know-how进行整合（若已有业务范围框定，则按业务范围）

In [ ]:
import json
import json5
import time
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import os
from prompts import compression_v0, safe_parse_json

# ── 1. 加载一级提炼结果 ──────────────────────────────────────────────────────
with open("./kh_results_zqrz317.json", "r", encoding="utf-8") as f:
    kh_data = json.load(f)

# 只保留成功且 Know_How 非空的条目，按 index 排序
valid_items = sorted(
    [
        {"index": v["index"], "Know_How": v["Know_How"]}
        for v in kh_data.values()
        if v.get("status") == "success" and v.get("Know_How", "").strip()
    ],
    key=lambda x: x["index"]
)
print(f"有效 Know_How 总数: {len(valid_items)}")

# ── 2. 切批（每 20 条一批）───────────────────────────────────────────────────
BATCH_SIZE = 20
batches = [valid_items[i : i + BATCH_SIZE] for i in range(0, len(valid_items), BATCH_SIZE)]
print(f"共 {len(batches)} 个批次，每批最多 {BATCH_SIZE} 条")

# ── 3. 辅助函数 ───────────────────────────────────────────────────────────────
COMPRESSION_OUTPUT = "./kh_compression_zqrz317.json"
compression_lock = Lock()


def format_batch_snippets(batch):
    """将一批 Know_How 格式化为 compression_v0 所需的输入文本。"""
    parts = []
    for i, item in enumerate(batch, 1):
        parts.append(f"【片段 {i}】（原始 index: {item['index']}）\n{item['Know_How']}")
    return "\n\n---\n\n".join(parts)


def _update_json_file_compression(file_path, key, value):
    """线程安全地更新 JSON 文件（读 → 改 → 写）。"""
    data_dict = {}
    if os.path.exists(file_path):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data_dict = json.load(f)
        except (json.JSONDecodeError, IOError):
            data_dict = {}
    data_dict[key] = value
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data_dict, f, ensure_ascii=False, indent=2)


# ── 4. 单批次处理（带重试）────────────────────────────────────────────────────
def process_compression_batch(batch_idx, batch, output_file, max_retries=5):
    """对单个批次调用 compression_v0 做知识整合，失败时指数退避重试。"""
    retry_count = 0
    base_wait = 3
    last_error_msg = ""

    while True:
        if retry_count > max_retries:
            error_result = {
                "batch_index": batch_idx,
                "source_indices": [item["index"] for item in batch],
                "item_count": len(batch),
                "status": "failed",
                "error": last_error_msg,
                "retry_count": retry_count,
            }
            with compression_lock:
                _update_json_file_compression(output_file, str(batch_idx), error_result)
            print(f"[Failed] 批次 {batch_idx} 超过最大重试次数 ({max_retries})")
            return batch_idx, "failed", None

        try:
            if retry_count > 0:
                print(f"[Retry {retry_count}] 批次 {batch_idx} 第 {retry_count + 1} 次尝试...")

            snippets_text = format_batch_snippets(batch)
            response = qwen(compression_v0(f=snippets_text))

            try:
                content = safe_parse_json(response["content"])
            except Exception as json_err:
                raise Exception(
                    f"JSON 解析失败（含兜底策略）: {json_err}"
                )

            if "Final_Know_How" not in content:
                raise Exception("响应缺少必需字段 'Final_Know_How'")

            result = {
                "batch_index": batch_idx,
                "source_indices": [item["index"] for item in batch],
                "item_count": len(batch),
                "Synthesis_Summary": content.get("Synthesis_Summary", ""),
                "Final_Know_How": content["Final_Know_How"],
                "status": "success",
                "retry_count": retry_count,
            }

            with compression_lock:
                _update_json_file_compression(output_file, str(batch_idx), result)

            suffix = f"（历经 {retry_count} 次重试）" if retry_count > 0 else ""
            print(f"[Success] 批次 {batch_idx + 1}/{len(batches)} 完成（{len(batch)} 条知识）{suffix}")
            return batch_idx, "success", result

        except Exception as e:
            retry_count += 1
            last_error_msg = str(e)
            wait_time = min(base_wait * (2 ** (retry_count - 1)), 60)
            print(f"[Error] 批次 {batch_idx} 第 {retry_count} 次失败: {last_error_msg[:150]}")
            print(f"[Wait] 等待 {wait_time} 秒后重试...")
            time.sleep(wait_time)


# ── 5. 多线程批量整合（支持断点续传）─────────────────────────────────────────
def run_compression(batches, max_workers=3, output_file=COMPRESSION_OUTPUT):
    existing_data = {}
    if os.path.exists(output_file):
        try:
            with open(output_file, "r", encoding="utf-8") as f:
                existing_data = json.load(f)
            print(f"📂 发现已有进度文件，包含 {len(existing_data)} 个批次记录，自动续传")
        except Exception:
            pass

    pending = [
        (i, b)
        for i, b in enumerate(batches)
        if str(i) not in existing_data
        or existing_data.get(str(i), {}).get("status") != "success"
    ]

    completed = len(batches) - len(pending)
    print(f"总批次: {len(batches)}，已完成: {completed}，待处理: {len(pending)}，并发数: {max_workers}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(process_compression_batch, i, b, output_file): i
            for i, b in pending
        }

        for future in as_completed(future_to_idx):
            batch_idx = future_to_idx[future]
            try:
                _, status, _ = future.result()
                if status == "success":
                    completed += 1
                    print(f"✅ 进度: {completed}/{len(batches)} ({completed / len(batches) * 100:.1f}%)")
                else:
                    print(f"❌ 批次 {batch_idx} 最终失败")
            except Exception as e:
                print(f"💥 批次 {batch_idx} 处理异常: {e}")

    print(f"\n🏁 全部完成！结果保存于: {output_file}")


# ── 6. 启动 ───────────────────────────────────────────────────────────────────
run_compression(batches, max_workers=5)

# 三级提炼：对二级提炼后的整合know-how，做逻辑合并、去重

In [ ]:
import json
import json5
import time
import traceback
import os
from prompts import merge_v0, shrink_v0, safe_parse_json

# -- 配置 --
COMPRESSION_FILE  = "./kh_compression_zqrz317.json"
PROGRESS_FILE     = "./kh_merge_progress_zqrz317.json"
FINAL_OUTPUT_FILE = "./kh_final_zqrz317.json"

MAX_KH_LEN    = 7000   # 超过此字符数则在合并前先收缩
TARGET_KH_LEN = 5000   # shrink 目标字符数
FINAL_MAX_LEN = 10000  # 最终输出强制不超过此字符数

# -- 1. 加载二级提炼结果，按 batch_index 排序 --
with open(COMPRESSION_FILE, "r", encoding="utf-8") as f:
    compression_data = json.load(f)

batches_l2 = sorted(
    [v for v in compression_data.values()
     if v.get("status") == "success" and v.get("Final_Know_How", "").strip()],
    key=lambda x: x["batch_index"]
)
print(f"二级提炼有效批次: {len(batches_l2)} 个")

# -- 2. 断点续传 --
current_kh = ""
start_from = 0

if os.path.exists(PROGRESS_FILE):
    try:
        with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
            progress = json.load(f)
        last_done  = progress.get("last_merged_batch", -1)
        current_kh = progress.get("current_kh", "")
        start_from = last_done + 1
        print(f"发现进度存档，已完成第 0~{last_done} 批，从第 {start_from} 批继续")
        print(f"  当前 KH 长度: {len(current_kh)} 字符")
    except Exception:
        print("进度文件损坏，从头开始")

# -- 3. 辅助函数 --
def save_progress(batch_idx, kh_text):
    snapshot = {"last_merged_batch": batch_idx, "current_kh": kh_text}
    with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
        json.dump(snapshot, f, ensure_ascii=False, indent=2)

def shrink_kh(kh_text, target_chars, label=""):
    """调用 shrink_v0 将 kh_text 压缩到 target_chars 字符以内，失败时返回原文。"""
    for attempt in range(1, 4):
        try:
            resp = qwen(shrink_v0(kh_text, target_chars=target_chars))
            parsed = safe_parse_json(resp["content"])
            shrunk = parsed.get("Shrunken_Know_How", "")
            if not shrunk.strip():
                raise ValueError("Shrunken_Know_How 为空")
            log = parsed.get("Shrink_Log", "")
            print(f"  [Shrink] {label}压缩: {len(kh_text)}->{len(shrunk)} 字符 | {log[:80]}")
            return shrunk
        except Exception as e:
            wait = 3 * (2 ** (attempt - 1))
            print(f"  [Shrink] {label}第 {attempt} 次失败: {str(e)[:100]}，{wait}s 后重试...")
            time.sleep(wait)
    print(f"  [Shrink] {label}全部失败，保留原文（长度 {len(kh_text)}）")
    return kh_text

# -- 4. 累增式合并主循环 --
total = len(batches_l2)
print(f"\n开始累增式合并，共 {total} 个批次，从第 {start_from} 批处理...")
print(f"长度策略：>{MAX_KH_LEN} 字符先收缩至 {TARGET_KH_LEN}，最终强制 <={FINAL_MAX_LEN}\n")

for i in range(start_from, total):
    batch       = batches_l2[i]
    increment   = batch["Final_Know_How"]
    src_indices = batch.get("source_indices", [])

    print(f"--- 第 {i+1}/{total} 批（src: {src_indices}，当前 KH: {len(current_kh)} 字符）---")

    # 长度检测：超限则先收缩 current_kh
    if len(current_kh) > MAX_KH_LEN:
        print(f"  [!] KH 超过 {MAX_KH_LEN} 字符，触发收缩...")
        current_kh = shrink_kh(current_kh, target_chars=TARGET_KH_LEN,
                                label=f"第{i+1}批前置-")

    max_retries = 5
    retry_count = 0
    base_wait   = 3

    while True:
        if retry_count > max_retries:
            print(f"  [Failed] 第 {i+1} 批超过最大重试次数，跳过")
            break
        try:
            if retry_count > 0:
                print(f"  [Retry {retry_count}] 第 {i+1} 批第 {retry_count+1} 次尝试...")

            response = qwen(merge_v0(existing=current_kh, increment=increment))

            try:
                content = safe_parse_json(response["content"])
            except Exception as je:
                raise Exception(f"JSON 解析失败: {je}")

            if "Merged_Know_How" not in content:
                raise Exception("响应缺少字段 Merged_Know_How")

            current_kh = content["Merged_Know_How"]
            merge_log  = content.get("Merge_Log", "")

            save_progress(i, current_kh)

            suffix = f"（历经 {retry_count} 次重试）" if retry_count > 0 else ""
            print(f"  [OK] 合并成功{suffix}，KH 长度: {len(current_kh)} 字符")
            print(f"  Merge_Log: {merge_log[:120]}..." if len(merge_log) > 120 else f"  Merge_Log: {merge_log}")
            break

        except Exception as e:
            retry_count += 1
            wait_time = min(base_wait * (2 ** (retry_count - 1)), 60)
            print(f"  [Error] 第 {i+1} 批第 {retry_count} 次失败: {str(e)[:150]}")
            print(f"  [Wait] 等待 {wait_time} 秒后重试...")
            time.sleep(wait_time)

# -- 5. 最终长度保障：强制压缩到 FINAL_MAX_LEN --
print(f"\n合并完成，最终 KH 长度: {len(current_kh)} 字符")
if len(current_kh) > FINAL_MAX_LEN:
    print(f"[!] 超过最终上限 {FINAL_MAX_LEN} 字符，执行最终压缩...")
    current_kh = shrink_kh(current_kh, target_chars=FINAL_MAX_LEN - 500, label="最终-")
    print(f"  最终压缩后长度: {len(current_kh)} 字符")

# -- 6. 输出最终 JSON --
final_result = {
    "status": "success",
    "total_l2_batches_merged": total,
    "final_kh_length": len(current_kh),
    "Final_Know_How": current_kh
}

with open(FINAL_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_result, f, ensure_ascii=False, indent=2)

print(f"\n三级提炼完成！最终知识库已保存至: {FINAL_OUTPUT_FILE}")
print(f"最终 KH 字符数: {len(current_kh)}")
print(f"\n--- 最终 Know-How 预览（前 500 字符）---")
print(current_kh[:500], "..." if len(current_kh) > 500 else "")


# 表格拼接

In [ ]:
import json
with open("kh_results_zqrz317.json", "r", encoding="utf-8") as f:
    level1 = json.load(f)

In [ ]:
# 代码已合并至下方单元格，此处无需执行
pass

# ── 4. 导出 Excel（pandas 初步写入） ─────────────────────────────────────────
output_path = "output_kh_zqrz317.xlsx"
data.to_excel(output_path, index=False, engine="openpyxl")

# ── 5. 重新加载，进行合并单元格处理 ──────────────────────────────────────────
wb = openpyxl.load_workbook(output_path)
ws = wb.active

# 获取各 KH 列的列号（1-based）
header = [cell.value for cell in ws[1]]
col_l2 = header.index("二级提炼KH") + 1
col_l3 = header.index("三级提炼KH") + 1

# 构建 data 行标签 → Excel 行号（Excel 行 2 对应 data 第 0 个元素，行 1 是表头）
index_list = list(data.index)
label_to_xl = {label: pos + 2 for pos, label in enumerate(index_list)}

# ── 5a. 合并二级 KH 列：同一 level2 batch 的行合并为一个单元格 ────────────────
l2_batch_xl_rows = {}
for label, xl_row in label_to_xl.items():
    l2_key = row_to_l2_key.get(label)
    if l2_key is not None:
        l2_batch_xl_rows.setdefault(l2_key, []).append(xl_row)

for l2_key, xl_rows in sorted(l2_batch_xl_rows.items()):
    if len(xl_rows) <= 1:
        continue
    xl_rows.sort()
    ws.merge_cells(
        start_row=xl_rows[0], start_column=col_l2,
        end_row=xl_rows[-1], end_column=col_l2
    )
    ws.cell(row=xl_rows[0], column=col_l2).alignment = Alignment(
        wrap_text=True, vertical="top"
    )

# ── 5b. 合并三级 KH 列：全部数据行合并为一个单元格 ───────────────────────────
n = len(data)
if n > 1:
    ws.merge_cells(start_row=2, start_column=col_l3, end_row=n + 1, end_column=col_l3)
    ws.cell(row=2, column=col_l3).alignment = Alignment(wrap_text=True, vertical="top")

wb.save(output_path)
print(f"✅ 已保存至 {output_path}（共 {n} 行数据，二级批次 {len(l2_batch_xl_rows)} 组合并）")


In [ ]:
import json
import openpyxl
from openpyxl.styles import Alignment

# ── 1. 写入一级 KH（每行独立提炼结果） ───────────────────────────────────────
for i in range(len(level1)):
    l1_data = level1[str(i)]
    know_how = l1_data['Know_How']
    logic = l1_data['Logic_Diagnosis']
    data.loc[i, "know-how"] = know_how
    data.loc[i, "Logic_Diagnosis"] = logic

# ── 2. 加载二级（batch 压缩）和三级（最终合并）数据 ──────────────────────────
with open("kh_compression_zqrz317.json", "r", encoding="utf-8") as f:
    level2 = json.load(f)

with open("kh_merge_progress_zqrz317.json", "r", encoding="utf-8") as f:
    level3_data = json.load(f)
level3_kh = level3_data["current_kh"]

# ── 3. 构建 原始行索引 → 二级批次 key 的映射 ─────────────────────────────────
row_to_l2_key = {}
for k, v in level2.items():
    for row_idx in v["source_indices"]:
        row_to_l2_key[row_idx] = k

# ── 4. 向 data 写入二级和三级 KH 列 ──────────────────────────────────────────
data["二级提炼KH"] = None
data["三级提炼KH"] = level3_kh  # 所有行相同

for row_idx in data.index:
    l2_key = row_to_l2_key.get(row_idx)
    if l2_key is not None:
        data.at[row_idx, "二级提炼KH"] = level2[l2_key]["Final_Know_How"]

print(f"二级 KH 覆盖行数: {data['二级提炼KH'].notna().sum()} / {len(data)}")

# ── 5. 导出 Excel（pandas 初步写入） ─────────────────────────────────────────
output_path = "output_kh_zqrz317.xlsx"
data.to_excel(output_path, index=False, engine="openpyxl")

# ── 6. 重新加载，进行合并单元格处理 ──────────────────────────────────────────
wb = openpyxl.load_workbook(output_path)
ws = wb.active

# 获取各 KH 列的列号（1-based）
header = [cell.value for cell in ws[1]]
col_l2 = header.index("二级提炼KH") + 1
col_l3 = header.index("三级提炼KH") + 1

# 构建 data 行标签 → Excel 行号（Excel 行 2 对应 data 第 0 个元素，行 1 是表头）
index_list = list(data.index)
label_to_xl = {label: pos + 2 for pos, label in enumerate(index_list)}

# ── 6a. 合并二级 KH 列：同一 level2 batch 的行合并为一个单元格 ────────────────
l2_batch_xl_rows = {}
for label, xl_row in label_to_xl.items():
    l2_key = row_to_l2_key.get(label)
    if l2_key is not None:
        l2_batch_xl_rows.setdefault(l2_key, []).append(xl_row)

for l2_key, xl_rows in sorted(l2_batch_xl_rows.items()):
    if len(xl_rows) <= 1:
        continue
    xl_rows.sort()
    ws.merge_cells(
        start_row=xl_rows[0], start_column=col_l2,
        end_row=xl_rows[-1], end_column=col_l2
    )
    ws.cell(row=xl_rows[0], column=col_l2).alignment = Alignment(
        wrap_text=True, vertical="top"
    )

# ── 6b. 合并三级 KH 列：全部数据行合并为一个单元格 ───────────────────────────
n = len(data)
if n > 1:
    ws.merge_cells(start_row=2, start_column=col_l3, end_row=n + 1, end_column=col_l3)
    ws.cell(row=2, column=col_l3).alignment = Alignment(wrap_text=True, vertical="top")

wb.save(output_path)
print(f"✅ 已保存至 {output_path}（共 {n} 行数据，二级批次 {len(l2_batch_xl_rows)} 组合并）")